In [5]:
import os

# Clone the GitHub repository to access the data file
repo_url = 'https://github.com/d1vyesh-27/flyrank-ai-internship-ml'
repo_name = repo_url.split('/')[-1]

if not os.path.exists(repo_name):
    !git clone {repo_url}
else:
    print(f"Repository '{repo_name}' already cloned.")

# Change the current working directory to the cloned repository
os.chdir(repo_name)

Cloning into 'flyrank-ai-internship-ml'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 103 (delta 22), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 1.84 MiB | 8.56 MiB/s, done.
Resolving deltas: 100% (22/22), done.


# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking (scoring). We're not classifying pages into categories (that would be classification), not grouping them unsupervised (clustering), and not just predicting a raw number. The model produces a continuous opportunity score per page, then we rank by that score to decide which pages an editor should look at first. The lane's output is a ranked list, so ranking is the right task type.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: CTR — directly observed in the performance data (a measured rate, not a rule). The model predicts expected CTR given features like position tier, impressions, content type, etc. Then the opportunity score is the residual (actual CTR - predicted CTR): pages well below their expected CTR rank highest for review. The target is observed, not defined, which satisfies the "no defined rules" rule.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@K. We define a "true opportunity" page as one whose actual CTR is significantly below its position-tier expected CTR (e.g., bottom 10% of residuals within its tier). Then precision@K measures what fraction of our top K ranked recommendations are genuine underperformers. This is measurable offline from the same data, and it directly reflects whether the ranked queue surfaces the right candidates — which is the model's job. The real-world follow-up (does editor action actually lift CTR?) is a separate measurement we cannot evaluate from a static dataset

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page. Each row is a single page's trailing-90-day performance snapshot. This is the right grain because the opportunity score is per-page — we're deciding which individual pages deserve editorial review.

In [7]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

cols = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "position_tier",
    "content_type",
    "content_age_days"
]

df[cols].head()

,content_id,impressions_90d,clicks_90d,ctr,avg_position,position_tier,content_type,content_age_days
0,content_304f48230142,3803,29,0.76,10.6,striking,keyword article,187
1,content_a1fb4e703a9e,15320,7,0.05,20.3,page_3_5,keyword article,445
2,content_9aa793d4d895,12581,11,0.09,36.5,page_3_5,keyword article,141
3,content_331d6c4de07b,11751,58,0.49,6.2,page_1,keyword article,463
4,content_d99b7a2d90ca,19140,24,0.13,44.0,page_3_5,keyword article,263


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

ML captures interactions between signals that are too tangled for fixed rules. A page's expected CTR depends on position tier, content type, impression volume, freshness, and query mix — all interacting. The same CTR can mean "urgent fix needed" for one page and "doing fine" for another depending on context. An if-statement would need dozens of crossed thresholds; a model learns these patterns from the data instead.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.